In [1]:
from setup import bedrock_tool
import chromadb
from pathlib import Path
from agents import Agent, Runner, function_tool, trace

✅ AWS credentials found
✅ OpenAI credentials found
✅ EXA credentials found


In [2]:
chroma_client = chromadb.PersistentClient("../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_qna")

In [3]:
results = nutrition_db.query(query_texts=["ocular"], n_results=2)
for i, doc in enumerate(results["documents"][0]):   
    print(doc)
    print("\n")

Question: How might a deficiency in vitamin A affect the eyes specifically?
        Answer: A shortage of vitamin A can result in ocular complications such as night blindness, xerophthalmia, and corneal ulcers that may lead to serious consequences if not treated.

        This Q&A pair provides information about nutrition and health topics.


Question: What is the clinical significance of Bitot’s spots?
        Answer: The presence of Bitot's spots signifies vitamin A deficiency. These are small, creamy-colored patches that appear on the white part of the eye.

        This Q&A pair provides information about nutrition and health topics.




In [4]:
@function_tool
def qna_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up answers on questions related to nutrition.

    Args:
        query: The question to look up.
        max_results: The maximum number of answers to return.

    Returns:
        A string containing the answer on a nutrition question.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No answer found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        question_item = metadata["question"].title()
        answer = metadata["answer"]

        formatted_results.append(
            f"{question_item} (): {answer} "
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

In [5]:
qna_agent = Agent(
    name="Q and A Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out replies to user's questions.
    You give concise answers.
    If you need to look up replies, use the qna_lookup_tool.
    """,
    model="litellm/bedrock/eu.amazon.nova-lite-v1:0",
    tools=[bedrock_tool(qna_lookup_tool.__dict__)],
)

In [8]:
with trace("Q and A Nutrition Assistant with RAG") as t:
    result = await Runner.run(
        qna_agent,
        "What is one of the clinical forms of Protein Energy Malnutriton?",
    )

print(f"Output: {result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

Output: 
Protein Energy Malnutrition (PEM) has several clinical forms, including:
1. Kwashiorkor: This form is characterized by edema and is usually seen in children who have a diet deficient in protein but adequate in calories.
2. Marasmus: This form is characterized by severe muscle wasting and is usually seen in children who have a diet deficient in both protein and calories.
3. Marasmic Kwashiorkor: This form is a combination of both Kwashiorkor and Marasmus.

Trace URL: https://platform.openai.com/logs/trace?trace_id=trace_e341e513e2d04192952dd35d681325e2
